In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Model Experiments and Hyperparameter Tuning\n",
    "\n",
    "Experiment with different models, hyperparameters, and techniques to improve prediction accuracy.\n",
    "\n",
    "## Experiments\n",
    "1. Hyperparameter tuning\n",
    "2. Different model architectures\n",
    "3. Ensemble methods\n",
    "4. Cross-validation strategies\n",
    "5. Feature subset testing"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "sys.path.append(os.path.dirname(os.getcwd()))\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score\n",
    "from sklearn.ensemble import GradientBoostingRegressor, VotingRegressor, StackingRegressor\n",
    "from sklearn.linear_model import Ridge, Lasso, ElasticNet\n",
    "from sklearn.tree import DecisionTreeRegressor\n",
    "from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "from src.train_model import prepare_features_target, split_data\n",
    "from src.evaluate import calculate_metrics\n",
    "from utils.config import DATA_PROCESSED_DIR, STOCK_SYMBOL\n",
    "\n",
    "plt.style.use('seaborn-v0_8')\n",
    "%matplotlib inline\n",
    "\n",
    "# For reproducibility\n",
    "np.random.seed(42)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load Data and Prepare"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "filepath = os.path.join(DATA_PROCESSED_DIR, f\"{STOCK_SYMBOL}_processed.csv\")\n",
    "df = pd.read_csv(filepath)\n",
    "\n",
    "X, y, feature_cols = prepare_features_target(df)\n",
    "X_train, X_test, y_train, y_test = split_data(X, y)\n",
    "\n",
    "print(f\"Training set: {X_train.shape}\")\n",
    "print(f\"Test set: {X_test.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Experiment 1: XGBoost Hyperparameter Tuning"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from xgboost import XGBRegressor\n",
    "\n",
    "# Define parameter grid\n",
    "param_grid = {\n",
    "    'n_estimators': [50, 100, 200],\n",
    "    'max_depth': [3, 5, 7],\n",
    "    'learning_rate': [0.01, 0.05, 0.1],\n",
    "    'subsample': [0.8, 0.9, 1.0]\n",
    "}\n",
    "\n",
    "# Create base model\n",
    "xgb_base = XGBRegressor(random_state=42)\n",
    "\n",
    "# Randomized search (faster than GridSearch)\n",
    "random_search = RandomizedSearchCV(\n",
    "    xgb_base,\n",
    "    param_distributions=param_grid,\n",
    "    n_iter=10,  # Number of parameter settings sampled\n",
    "    cv=3,\n",
    "    scoring='neg_mean_squared_error',\n",
    "    random_state=42,\n",
    "    n_jobs=-1,\n",
    "    verbose=1\n",
    ")\n",
    "\n",
    "print(\"Starting hyperparameter tuning...\")\n",
    "random_search.fit(X_train, y_train)\n",
    "\n",
    "print(\"\\n✅ Best parameters found:\")\n",
    "print(random_search.best_params_)\n",
    "\n",
    "print(\"\\n📊 Best cross-validation score:\")\n",
    "print(f\"RMSE: {np.sqrt(-random_search.best_score_):.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Evaluate best model\n",
    "best_xgb = random_search.best_estimator_\n",
    "y_pred = best_xgb.predict(X_test)\n",
    "\n",
    "metrics = calculate_metrics(y_test, y_pred)\n",
    "print(\"\\nTest Set Performance (Tuned XGBoost):\")\n",
    "for metric, value in metrics.items():\n",
    "    print(f\"{metric:8s}: {value:12.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Experiment 2: Compare Multiple Models"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.svm import SVR\n",
    "from sklearn.neighbors import KNeighborsRegressor\n",
    "from sklearn.ensemble import AdaBoostRegressor\n",
    "\n",
    "# Define models to test\n",
    "models = {\n",
    "    'Ridge': Ridge(alpha=1.0),\n",
    "    'Lasso': Lasso(alpha=1.0),\n",
    "    'ElasticNet': ElasticNet(alpha=1.0),\n",
    "    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),\n",
    "    'KNN': KNeighborsRegressor(n_neighbors=5),\n",
    "    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),\n",
    "    'AdaBoost': AdaBoostRegressor(n_estimators=50, random_state=42)\n",
    "}\n",
    "\n",
    "results = []\n",
    "\n",
    "for name, model in models.items():\n",
    "    print(f\"Training {name}...\")\n",
    "    model.fit(X_train, y_train)\n",
    "    \n",
    "    # Predictions\n",
    "    y_train_pred = model.predict(X_train)\n",
    "    y_test_pred = model.predict(X_test)\n",
    "    \n",
    "    # Metrics\n",
    "    train_r2 = r2_score(y_train, y_train_pred)\n",
    "    test_r2 = r2_score(y_test, y_test_pred)\n",
    "    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))\n",
    "    test_mae = mean_absolute_error(y_test, y_test_pred)\n",
    "    \n",
    "    results.append({\n",
    "        'Model': name,\n",
    "        'Train_R2': train_r2,\n",
    "        'Test_R2': test_r2,\n",
    "        'Test_RMSE': test_rmse,\n",
    "        'Test_MAE': test_mae\n",
    "    })\n",
    "\n",
    "results_df = pd.DataFrame(results).sort_values('Test_R2', ascending=False)\n",
    "print(\"\\n\" + \"=\"*70)\n",
    "print(\"MODEL COMPARISON RESULTS\")\n",
    "print(\"=\"*70)\n",
    "print(results_df.to_string(index=False))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize model comparison\n",
    "fig, axes = plt.subplots(1, 2, figsize=(15, 5))\n",
    "\n",
    "# R2 scores\n",
    "axes[0].barh(results_df['Model'], results_df['Test_R2'], color='steelblue')\n",
    "axes[0].set_xlabel('Test R² Score')\n",
    "axes[0].set_title('Model Comparison - R² Score')\n",
    "axes[0].grid(True, alpha=0.3, axis='x')\n",
    "\n",
    "# RMSE\n",
    "axes[1].barh(results_df['Model'], results_df['Test_RMSE'], color='coral')\n",
    "axes[1].set_xlabel('Test RMSE')\n",
    "axes[1].set_title('Model Comparison - RMSE')\n",
    "axes[1].grid(True, alpha=0.3, axis='x')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Experiment 3: Ensemble Methods"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.ensemble import RandomForestRegressor\n",
    "\n",
    "# Voting Ensemble\n",
    "voting_model = VotingRegressor([\n",
    "    ('rf', RandomForestRegressor(n_estimators=50, random_state=42)),\n",
    "    ('xgb', XGBRegressor(n_estimators=50, random_state=42)),\n",
    "    ('gb', GradientBoostingRegressor(n_estimators=50, random_state=42))\n",
    "])\n",
    "\n",
    "print(\"Training Voting Ensemble...\")\n",
    "voting_model.fit(X_train, y_train)\n",
    "y_pred_voting = voting_model.predict(X_test)\n",
    "\n",
    "metrics_voting = calculate_metrics(y_test, y_pred_voting)\n",
    "print(\"\\nVoting Ensemble Performance:\")\n",
    "for metric, value in metrics_voting.items():\n",
    "    print(f\"{metric:8s}: {value:12.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Stacking Ensemble\n",
    "estimators = [\n",
    "    ('rf', RandomForestRegressor(n_estimators=50, random_state=42)),\n",
    "    ('xgb', XGBRegressor(n_estimators=50, random_state=42))\n",
    "]\n",
    "\n",
    "stacking_model = StackingRegressor(\n",
    "    estimators=estimators,\n",
    "    final_estimator=Ridge(),\n",
    "    cv=3\n",
    ")\n",
    "\n",
    "print(\"\\nTraining Stacking Ensemble...\")\n",
    "stacking_model.fit(X_train, y_train)\n",
    "y_pred_stacking = stacking_model.predict(X_test)\n",
    "\n",
    "metrics_stacking = calculate_metrics(y_test, y_pred_stacking)\n",
    "print(\"\\nStacking Ensemble Performance:\")\n",
    "for metric, value in metrics_stacking.items():\n",
    "    print(f\"{metric:8s}: {value:12.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Experiment 4: Feature Subset Selection"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Test with different number of features\n",
    "from sklearn.feature_selection import SelectKBest, f_regression\n",
    "\n",
    "feature_counts = [5, 10, 15, 20, 30]\n",
    "subset_results = []\n",
    "\n",
    "for k in feature_counts:\n",
    "    print(f\"\\nTesting with top {k} features...\")\n",
    "    \n",
    "    # Select top k features\n",
    "    selector = SelectKBest(score_func=f_regression, k=k)\n",
    "    X_train_selected = selector.fit_transform(X_train, y_train)\n",
    "    X_test_selected = selector.transform(X_test)\n",
    "    \n",
    "    # Train model\n",
    "    model = XGBRegressor(n_estimators=100, random_state=42)\n",
    "    model.fit(X_train_selected, y_train)\n",
    "    \n",
    "    # Evaluate\n",
    "    y_pred = model.predict(X_test_selected)\n",
    "    r2 = r2_score(y_test, y_pred)\n",
    "    rmse = np.sqrt(mean_squared_error(y_test, y_pred))\n",
    "    \n",
    "    subset_results.append({\n",
    "        'Num_Features': k,\n",
    "        'Test_R2': r2,\n",
    "        'Test_RMSE': rmse\n",
    "    })\n",
    "\n",
    "subset_df = pd.DataFrame(subset_results)\n",
    "print(\"\\n\" + \"=\"*50)\n",
    "print(\"FEATURE SUBSET RESULTS\")\n",
    "print(\"=\"*50)\n",
    "print(subset_df.to_string(index=False))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot feature subset results\n",
    "fig, axes = plt.subplots(1, 2, figsize=(15, 5))\n",
    "\n",
    "axes[0].plot(subset_df['Num_Features'], subset_df['Test_R2'], \n",
    "            marker='o', linewidth=2, markersize=8)\n",
    "axes[0].set_xlabel('Number of Features')\n",
    "axes[0].set_ylabel('Test R² Score')\n",
    "axes[0].set_title('R² vs Number of Features')\n",
    "axes[0].grid(True, alpha=0.3)\n",
    "\n",
    "axes[1].plot(subset_df['Num_Features'], subset_df['Test_RMSE'], \n",
    "            marker='o', linewidth=2, markersize=8, color='coral')\n",
    "axes[1].set_xlabel('Number of Features')\n",
    "axes[1].set_ylabel('Test RMSE')\n",
    "axes[1].set_title('RMSE vs Number of Features')\n",
    "axes[1].grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Experiment 5: Cross-Validation Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.model_selection import cross_val_score, KFold\n",
    "\n",
    "# Test different CV strategies\n",
    "cv_strategies = {\n",
    "    'KFold-3': KFold(n_splits=3, shuffle=True, random_state=42),\n",
    "    'KFold-5': KFold(n_splits=5, shuffle=True, random_state=42),\n",
    "    'KFold-10': KFold(n_splits=10, shuffle=True, random_state=42)\n",
    "}\n",
    "\n",
    "model = XGBRegressor(n_estimators=100, random_state=42)\n",
    "\n",
    "cv_results = []\n",
    "\n",
    "for name, cv in cv_strategies.items():\n",
    "    print(f\"\\nRunning {name} cross-validation...\")\n",
    "    scores = cross_val_score(model, X_train, y_train, \n",
    "                            cv=cv, scoring='r2', n_jobs=-1)\n",
    "    \n",
    "    cv_results.append({\n",
    "        'Strategy': name,\n",
    "        'Mean_R2': scores.mean(),\n",
    "        'Std_R2': scores.std(),\n",
    "        'Min_R2': scores.min(),\n",
    "        'Max_R2': scores.max()\n",
    "    })\n",
    "\n",
    "cv_df = pd.DataFrame(cv_results)\n",
    "print(\"\\n\" + \"=\"*70)\n",
    "print(\"CROSS-VALIDATION RESULTS\")\n",
    "print(\"=\"*70)\n",
    "print(cv_df.to_string(index=False))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Experiment 6: Learning Curves"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.model_selection import learning_curve\n",
    "\n",
    "# Generate learning curves\n",
    "train_sizes, train_scores, val_scores = learning_curve(\n",
    "    XGBRegressor(n_estimators=100, random_state=42),\n",
    "    X_train, y_train,\n",
    "    train_sizes=np.linspace(0.1, 1.0, 10),\n",
    "    cv=3,\n",
    "    scoring='r2',\n",
    "    n_jobs=-1\n",
    ")\n",
    "\n",
    "train_mean = train_scores.mean(axis=1)\n",
    "train_std = train_scores.std(axis=1)\n",
    "val_mean = val_scores.mean(axis=1)\n",
    "val_std = val_scores.std(axis=1)\n",
    "\n",
    "# Plot learning curves\n",
    "plt.figure(figsize=(10, 6))\n",
    "plt.plot(train_sizes, train_mean, label='Training Score', \n",
    "         marker='o', linewidth=2)\n",
    "plt.fill_between(train_sizes, train_mean - train_std, \n",
    "                train_mean + train_std, alpha=0.2)\n",
    "\n",
    "plt.plot(train_sizes, val_mean, label='Validation Score', \n",
    "         marker='s', linewidth=2)\n",
    "plt.fill_between(train_sizes, val_mean - val_std, \n",
    "                val_mean + val_std, alpha=0.2)\n",
    "\n",
    "plt.xlabel('Training Set Size')\n",
    "plt.ylabel('R² Score')\n",
    "plt.title('Learning Curves - XGBoost')\n",
    "plt.legend(loc='best')\n",
    "plt.grid(True, alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Experiment Summary"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create summary of all experiments\n",
    "summary = {\n",
    "    'Best XGBoost Params': random_search.best_params_,\n",
    "    'Best Model': results_df.iloc[0]['Model'],\n",
    "    'Best R2 Score': results_df.iloc[0]['Test_R2'],\n",
    "    'Optimal Feature Count': subset_df.loc[subset_df['Test_R2'].idxmax(), 'Num_Features'],\n",
    "    'Best Ensemble': 'Voting' if metrics_voting['R2'] > metrics_stacking['R2'] else 'Stacking'\n",
    "}\n",
    "\n",
    "print(\"\\n\" + \"=\"*70)\n",
    "print(\" \"*20 + \"EXPERIMENT SUMMARY\")\n",
    "print(\"=\"*70)\n",
    "for key, value in summary.items():\n",
    "    print(f\"{key:25s}: {value}\")\n",
    "print(\"=\"*70)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Conclusions and Next Steps\n",
    "\n",
    "### Key Findings:\n",
    "1. **Best Model Configuration**: \n",
    "   - Document the best model and parameters found\n",
    "\n",
    "2. **Feature Selection**: \n",
    "   - Optimal number of features\n",
    "   - Most important features\n",
    "\n",
    "3. **Ensemble Performance**:\n",
    "   - Whether ensembles improved results\n",
    "\n",
    "4. **Overfitting Analysis**:\n",
    "   - Gap between training and validation scores\n",
    "   - Learning curve insights\n",
    "\n",
    "### Recommendations:\n",
    "1. Use the best configuration for production\n",
    "2. Consider regularization if overfitting detected\n",
    "3. Collect more data if underfitting\n",
    "4. Try deep learning models (LSTM) for time series\n",
    "\n",
    "### Next Experiments:\n",
    "- [ ] Test with different stocks\n",
    "- [ ] Try LSTM/GRU models\n",
    "- [ ] Add external features (news, sentiment)\n",
    "- [ ] Implement online learning\n",
    "- [ ] Test different time horizons"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}